In [ ]:
import os
import numpy as np
import json
import pandas as pd
import geopandas as gpd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import sys
# append the path of the parent directory
sys.path.append("..")
from utils.dol import *


SELECTED_BANDS = [0, 12, 13, 14]  # 1,13,14,15 → 0-based
TARGET_SIZE = 32
CACHE_DIR = "/app/data/datasets/debug/bush/cache"
MODEL_SAVE_PATH = "/app/data/datasets/debug/bush/models"
gdf_datas = gpd.read_file('/app/data/datasets/debug/bush/target_dol_zones_v1.gpkg',driver='GPKG')
gdf_datas.to_crs('EPSG:2154', inplace=True)
gdf_datas.head(5)


test_gdf = gdf_datas[gdf_datas.insee_com=='34035']
train_gdf = gdf_datas[~(gdf_datas.insee_com=='34035')]

train_gdf, val_gdf = train_test_split(
    train_gdf,
    test_size=0.2,
    stratify=train_gdf.target_control,
    random_state=42
)
#train_gdf = train_gdf[:int(len(train_gdf)/3*2)]
print(f"trainset size : {len(train_gdf)} - % target control balance : {np.sum(train_gdf.target_control.tolist())/len(train_gdf)}")
print(f"valset size : {len(val_gdf)} - % target control balance : {np.sum(val_gdf.target_control.tolist())/len(val_gdf)}")
print(f"test size : {len(test_gdf)} - % target control balance : {np.sum(test_gdf.target_control.tolist())/len(test_gdf)}")

raster_crs = 'EPSG:2154'

In [ ]:

gdf_forest_zones = gpd.read_file('/app/data/datasets/debug/bush/ocsge_forests_clean_types_v3.gpkg',driver='GPKG')
gdf_water_zones = gpd.read_file('/app/data/datasets/debug/bush/COURS_D_EAU.shp')
gdf_forests_zones = gdf_forest_zones[gdf_forest_zones.forest_type==1]

X_train, y_train = get_or_build_xy(train_gdf, "train", gdf_forests_zones, gdf_water_zones, gdf_u_zones, cache_dir=CACHE_DIR, debug=False)
X_val, y_val     = get_or_build_xy(val_gdf, "val", gdf_forests_zones, gdf_water_zones, gdf_u_zones, cache_dir=CACHE_DIR, debug=False)
X_test, y_test   = get_or_build_xy(test_gdf, "test", gdf_forests_zones, gdf_water_zones, gdf_u_zones, cache_dir=CACHE_DIR, debug=False)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

x_train_business_features = X_train[['has_contact_river_zone','has_contact_forest_zone','has_inhabited_building','has_building']]
x_train_ml_features = X_train.drop(columns=['has_contact_river_zone','has_contact_forest_zone'])

x_val_business_features = X_val[['has_contact_river_zone','has_contact_forest_zone','has_inhabited_building','has_building']]
x_val_ml_features = X_val.drop(columns=['has_contact_river_zone','has_contact_forest_zone'])

x_test_business_features = X_test[['has_contact_river_zone','has_contact_forest_zone','has_inhabited_building','has_building']]
x_test_ml_features = X_test.drop(columns=['has_contact_river_zone','has_contact_forest_zone'])

In [ ]:
X_train

In [ ]:

xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.01,


    scale_pos_weight=scale_pos_weight,

    objective="binary:logistic",
    eval_metric="aucpr",

    tree_method="hist",
    random_state=42,
    early_stopping_rounds = 100
)

xgb_model.fit(
    x_train_ml_features, y_train,
    eval_set=[(x_val_ml_features, y_val)],
    verbose=True
)

xgb_model.save_model(os.path.join(MODEL_SAVE_PATH,'dol_xgb_model_v1.json'))

val_proba = xgb_model.predict_proba(x_val_ml_features)[:, 1]

best_thresh, best_score = find_best_threshold(
    y_true=y_val,
    y_proba=val_proba,
    metric="f1"
)

print(f"Best threshold = {best_thresh:.3f}, F1 = {best_score:.3f}")

# save metadata
metadata = {
    "decision_threshold": best_thresh,
    "metric_optimized": "f1",
    "scale_pos_weight": scale_pos_weight
}

with open(os.path.join(MODEL_SAVE_PATH,'dol_xgb_model_v1_metadata.json'), "w") as f:
    json.dump(metadata, f)


In [ ]:
df_features_imp = pd.DataFrame(columns = ['features', 'importance'], data = np.asarray([x_val_ml_features.columns, xgb_model.feature_importances_]).T)
df_features_imp.sort_values()

In [ ]:

test_proba = xgb_model.predict_proba(x_test_ml_features)[:, 1]
test_gdf["proba_control"] = test_proba
test_gdf["pred_control"] = 0
test_gdf.loc[test_gdf["proba_control"] >= best_thresh,"pred_control"] = 1
test_gdf
test_results_gdf = postprocess_pred_control(test_gdf, x_test_business_features)

test_results_gdf.to_file(
    "/app/data/datasets/debug/bush/pred_small_dol_zones_xgb_13.gpkg",
    driver="GPKG"
)